# Finding events, and the rest of the library

The other notebooks assume you already know which event you want. This one is
about getting there: the search helpers in `pyoly.utils`, plus the
configuration and packaging details that sit underneath everything else.

In [1]:
import sys
from pathlib import Path

# Run straight from a clone, without installing the package first.
src = Path.cwd().parent / "src"
if src.is_dir():
    sys.path.insert(0, str(src))

import pyolymarket as pyoly

## The raw search

`polymarket_search` is Polymarket's own search endpoint, returned as-is. You
get the matching events plus a `pagination` block telling you how many there
are in total.

In [2]:
payload = pyoly.polymarket_search("super bowl", limit_per_type = 3)

print(list(payload))
print(payload["pagination"])

for event in payload["events"]:
    print(event["slug"])

['events', 'pagination']
{'hasMore': True, 'totalResults': 158}
super-bowl-winner-2024
was-the-super-bowl-rigged
super-bowl-ushers-first-song-in-halftime-show


`page` walks through the rest of them.

In [3]:
for event in pyoly.polymarket_search("super bowl", limit_per_type = 3,
                                     page = 2)["events"]:
    print(event["slug"])

super-bowl-gatorade-color
taylor-swift-super-bowl-parlay
will-taylor-swift-attend-the-super-bowl


Asking for tags and profiles adds two more keys to the response.

In [4]:
extras = pyoly.polymarket_search("super bowl", limit_per_type = 1,
                                 search_tags = True, search_profiles = True)

print(list(extras))
print([tag["label"] for tag in extras["tags"]])

['events', 'tags', 'profiles', 'pagination']
['Super Bowl']


Gamma silently ignores query parameters it does not recognize and answers 200
with an unfiltered result set, so a typo would look like bad ranking rather
than a mistake. The accepted names are pinned in the wrapper and anything else
raises:

In [5]:
try:
    pyoly.polymarket_search("super bowl", limit = 3)
except ValueError as error:
    print(error)

Unrecognized search parameter(s): limit. Accepted parameters are: ascending, cache, events_status, events_tag, exclude_tag_id, keep_closed_markets, limit_per_type, optimized, page, q, recurrence, search_profiles, search_tags, sort


In [6]:
sorted(pyoly.utils.SEARCH_PARAMS)

['ascending',
 'cache',
 'events_status',
 'events_tag',
 'exclude_tag_id',
 'keep_closed_markets',
 'limit_per_type',
 'optimized',
 'page',
 'q',
 'recurrence',
 'search_profiles',
 'search_tags',
 'sort']

## Search that hands back Events

`polymarket_search_event` is the same endpoint wrapped in `Event` objects. It
is still one request in total: search hits arrive with their markets nested,
so nothing is re-fetched.

In [7]:
events = pyoly.polymarket_search_event("super bowl", results = 3)

for event in events:
    print(f"{len(event.markets):>3} markets  {event.title}")

 20 markets  Super Bowl Winner 2024
  1 markets  Was the Super Bowl rigged?
  5 markets  Super Bowl: Usher's first song in Halftime Show?


`events_status` picks between what is live and what has already settled.

In [8]:
for status in ("active", "resolved"):
    found = pyoly.polymarket_search_event("bitcoin", results = 3,
                                          events_status = status)
    print(status, "->", [e.title for e in found])

active -> ['What price will Bitcoin hit in August?', 'What price will Bitcoin hit in 2026?', 'Bitcoin above ___ on August 26?']
resolved -> ['Bitcoin above ___ on August 25?', 'Bitcoin above ___ on February 23?', 'Bitcoin Up or Down - April 2, 9:50PM-9:55PM ET']


`sort` reorders the hits by a field on the event, `ascending` flips the
direction.

In [9]:
for event in pyoly.polymarket_search_event("bitcoin", results = 3,
                                           sort = "volume", ascending = False):
    print(f"{event.data['volume']:>15,.0f}  {event.title}")

    407,170,412  MicroStrategy sells any Bitcoin by ___ ?
    188,753,254  What price will Bitcoin hit in 2025?
    121,274,308  What price will Bitcoin hit in February?


With `results = 1` you get a bare `Event` rather than a one-item list.

In [10]:
pyoly.polymarket_search_event("super bowl", results = 1)

Event(slug='super-bowl-winner-2024', id='903197')

## Searching the web instead

`ddg_search_event` runs the query through DuckDuckGo and turns any
polymarket.com event links it finds back into `Event` objects. It often reads
a vague query better than the native search does, but it depends on an outside
search engine, so it is the least predictable of the three: it can come back
empty, get rate-limited, or hand you a stale link whose slug 404s. Hence the
`try` below. It needs the optional `search` extra:

```shell
pip install pyolymarket[search]
```

`depth` is how many web results to look through, `max_results` how many events
to return.

In [11]:
try:
    print(pyoly.ddg_search_event("super bowl winner", depth = 10,
                                 max_results = 2))
except Exception as error:
    print(type(error).__name__, error)

[Event(slug='super-bowl-champion-2026-731', id='23656'), Event(slug='super-bowl-lx-mvp', id='189545')]


The third option, `embedded_search_event`, ranks pre-embedded events by
meaning rather than by keyword. It needs an embedding key and is covered in
`03_embedding_basics.ipynb`.

## Configuration

`pyoly.config` is a single shared object. Every credential is read from the
environment at the moment it is used, and the variable names themselves are
settable, which is what you change when your keys already live somewhere else.

In [12]:
print(pyoly.config.EMB_API_KEY_ENV)
print(pyoly.config.CLOB_API_KEY_ENV)
print(pyoly.config.CLOB_SECRET_ENV)
print(pyoly.config.CLOB_PASSPHRASE_ENV)
print(pyoly.config.CLOB_ADDRESS_ENV)
print(pyoly.config.CLOB_PRIVATE_KEY_ENV)

PYOLY_OPENAI_API_KEY
PYOLY_CLOB_API_KEY
PYOLY_CLOB_SECRET
PYOLY_CLOB_PASSPHRASE
PYOLY_CLOB_ADDRESS
PYOLY_CLOB_PRIVATE_KEY


In [13]:
pyoly.config.CLOB_API_KEY_ENV = "MY_SILLY_CLOB_API_KEY"
print(pyoly.config.CLOB_API_KEY_ENV)

pyoly.config.CLOB_API_KEY_ENV = "PYOLY_CLOB_API_KEY"

MY_SILLY_CLOB_API_KEY


Missing credentials are reported all at once, naming the variables you
actually have to set:

In [14]:
try:
    pyoly.config.clob_creds
except EnvironmentError as error:
    print(error)

Missing env var(s): PYOLY_CLOB_API_KEY, PYOLY_CLOB_SECRET, PYOLY_CLOB_PASSPHRASE, PYOLY_CLOB_ADDRESS. Authenticated CLOB reads need L2 credentials; derive them with pyolymarket.clob.derive_api_key().


Caching is off by default. `caching` takes a bool for the common case or one
of the level strings: `"csv"` writes only the human-readable copy, `"npy"`
only the parquet and vector files that get read back, `"on"` writes both.

In [15]:
print(pyoly.config.cache_level, "->", pyoly.config.caching)

pyoly.config.caching = True
print(pyoly.config.cache_level, "->", pyoly.config.caching)

try:
    pyoly.config.caching = "sometimes"
except ValueError as error:
    print(error)

pyoly.config.caching = False

off -> False
on -> True
Unrecongnized level argument: sometimes. level must be off, on, csv, npy


`CACHE_DIR` is read when it is used rather than at import, so it lands
relative to the working directory you actually have. Point it somewhere
explicit if you would rather it did not follow you around.

In [16]:
print(pyoly.config.CACHE_DIR)

/workspace/tests/___pyolymarket_cache___


## Optional dependencies

The Gamma and CLOB surfaces need nothing but `requests`. Embeddings, the disk
cache and the DuckDuckGo search each pull a heavier dependency, so those
modules are resolved on first attribute access instead of at import time. That
keeps `import pyolymarket` cheap, and keeps it free of side effects: importing
the cacher can kick off a full catalogue build.

`pyoly.utils` above was loaded by touching it. `pyoly.clob` has not been
touched in this notebook, so it is not loaded yet:

In [17]:
print("clob imported:", "pyolymarket.clob" in sys.modules)

pyoly.clob.INTERVALS

clob imported: False


('max', 'all', '1m', '1w', '1d', '6h', '1h')

In [18]:
print("clob imported:", "pyolymarket.clob" in sys.modules)

clob imported: True


A missing extra is reported at that first access, naming the one to install
rather than failing back at `import pyolymarket`. Everything reachable this
way:

In [19]:
print(pyoly.__all__)

['Event', 'Market', 'config', 'clob', 'cacher', 'utils', 'embedding_logic', 'embed', 'embed_list', 'polymarket_search', 'polymarket_search_event', 'ddg_search_event', 'embedded_search_event', 'EmbeddingError', 'PolymarketAPIError', 'PolymarketNotFoundError', 'PolymarketRateLimitError']


In [20]:
print(pyoly.__version__)

0.0.0+unknown


That version reads `0.0.0+unknown` when you run from a clone without
installing the package, since it comes from the installed distribution's
metadata.